In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational"
)
model = ChatHuggingFace(llm=llm)

d:\MY_study\AgenticAI_Tutorial\LangGraph_Tutorial\lkr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Create a state
class LLMState (TypedDict):
    question : str
    answer : str

In [21]:
def llm_qa(state: LLMState) -> LLMState:

    #extract the question from the state
    question = state['question']

    #form a prompt
    prompt = f"Answer the following question: {question}"

    #ask that question to the LLm
    answer = model.invoke(prompt).content

    #update the answer in the state
    state['answer'] = answer
    return state


In [22]:
#create our graph
graph=StateGraph(LLMState)

#add nodes
graph.add_node('llm_qa', llm_qa)

#add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#compile the graph into a workflow
workflow=graph.compile()

In [26]:
#execute the graph
initial_state = {'question': "What is the capital of India?"}
final_state = workflow.invoke(initial_state)
print(final_state)
print(final_state['answer'])

{'question': 'What is the capital of India?', 'answer': 'The capital of India is New Delhi.'}
The capital of India is New Delhi.
